# Feature Selection

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load data with pseudo labels
data_path = Path("../outputs/hrv_windows_final.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nPseudo label distribution:")
print(df['pseudo_label_smoothed'].value_counts().sort_index())

Data shape: (2541, 31)

Columns: ['dataset', 'subject_id', 'trial_id', 'window_index', 'window_start_sec', 'window_end_sec', 'sampling_rate', 'mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'nn50', 'pnn50', 'cv_rr', 'mean_rr_znorm', 'mean_hr_znorm', 'sdnn_znorm', 'rmssd_znorm', 'nn50_znorm', 'pnn50_znorm', 'cv_rr_znorm', 'cluster_kmeans', 'pseudo_kss_kmeans', 'pseudo_label_kmeans', 'cluster_gmm', 'pseudo_kss_gmm', 'pseudo_label_gmm', 'pseudo_kss', 'pseudo_label', 'pseudo_kss_smoothed', 'pseudo_label_smoothed']

Pseudo label distribution:
pseudo_label_smoothed
0    1616
1     925
Name: count, dtype: int64


In [4]:
# Identify HRV feature columns (exclude metadata and labels)
exclude_cols = {'dataset', 'subject_id', 'trial_id', 'window_index',
                'window_start_sec', 'window_end_sec', 'sampling_rate',
                'mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'nn50', 'pnn50', 'cv_rr',
                'cluster_kmeans', 'pseudo_kss_kmeans', 'pseudo_label_kmeans',
                'cluster_gmm', 'pseudo_kss_gmm', 'pseudo_label_gmm', 'pseudo_kss',
                'pseudo_label', 'pseudo_kss_smoothed', 'pseudo_label_smoothed'
        }
hrv_features = [col for col in df.columns if col not in exclude_cols]

print(f"HRV Features ({len(hrv_features)}): {hrv_features}\n")

HRV Features (7): ['mean_rr_znorm', 'mean_hr_znorm', 'sdnn_znorm', 'rmssd_znorm', 'nn50_znorm', 'pnn50_znorm', 'cv_rr_znorm']



In [6]:
print("=" * 80)
print("PATTERN ANALYSIS: Feature Separability by Pseudo Label")
print("=" * 80)

# Calculate separation quality: F-statistic for each feature
from scipy.stats import f_oneway

feature_separability = []

for feature in hrv_features:
    # Get groups of values for each pseudo label
    groups = [group[feature].values for name, group in df.groupby('pseudo_label_smoothed')]
    
    # Only compute if we have variance in groups
    if len(groups) > 1 and all(len(g) > 0 for g in groups):
        f_stat, p_value = f_oneway(*groups)
        feature_separability.append({
            'Feature': feature,
            'F-Statistic': f_stat,
            'P-Value': p_value,
            'Separability': 'High' if p_value < 0.001 else ('Medium' if p_value < 0.05 else 'Low')
        })

df_separability = pd.DataFrame(feature_separability).sort_values('F-Statistic', ascending=False)

print("\nFeature Separability Ranking (ordered by F-Statistic):")
print(df_separability.head(4).to_string(index=False))

PATTERN ANALYSIS: Feature Separability by Pseudo Label

Feature Separability Ranking (ordered by F-Statistic):
    Feature  F-Statistic       P-Value Separability
 sdnn_znorm  2290.798344  0.000000e+00         High
cv_rr_znorm  1958.213994  0.000000e+00         High
rmssd_znorm  1217.074354 3.452827e-218         High
pnn50_znorm   996.525828 8.531825e-185         High


## Decided features

- sdnn_znorm
- cv_rr_znorm
- rmssd_znorm
- pnn50_znorm

```
Feature Separability Ranking (ordered by F-Statistic):
    Feature  F-Statistic       P-Value Separability
 sdnn_znorm  2290.798344  0.000000e+00         High
cv_rr_znorm  1958.213994  0.000000e+00         High
rmssd_znorm  1217.074354 3.452827e-218         High
pnn50_znorm   996.525828 8.531825e-185         High
```